PRECISION/RECALL/F1-SCORE

In [ ]:
import pandas as pd
import re
import spacy


#Load the dataset
file_path="INSERT_FILE_PATH"
df=pd.read_excel(file_path)

# Load the English NLP grammar model
# Note: If this fails, make sure you ran: python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

def extract_smart_terms(text, tag_name):
    """
    Uses Natural Language Processing to find the tag and extract
    only the noun phrase immediately preceding it.
    """
    if pd.isna(text) or not isinstance(text, str):
        return []

    # Strip brackets for the regex search
    clean_tag = tag_name.strip('[]')

    # Regex: Grab up to 5 words before the tag (handles missing closing brackets)
    pattern = rf"((?:[\w\'\-]+\s*){{1,5}})\s*\[{clean_tag}(?:\]|\b)"
    matches = re.findall(pattern, text)

    extracted_terms = []

    for match in matches:
        # Clean up the matched string
        raw_phrase = match.strip()

        # Let spaCy analyze the grammar of the phrase
        doc = nlp(raw_phrase)

        # Extract the "Noun Chunks" (e.g., "cow urine", "the throat")
        noun_chunks = list(doc.noun_chunks)

        if noun_chunks:
            # Grab the last noun chunk right before the tag
            term = noun_chunks[-1].text.strip("'\"")

            # Remove leading grammatical filler words (determiners/pronouns)
            term_words = term.split()
            if term_words and term_words[0].lower() in ['the', 'a', 'an', 'some', 'any', 'my', 'your', 'his', 'her', 'its', 'our', 'their']:
                term = " ".join(term_words[1:])

            if term:
                extracted_terms.append(term)
        else:
            # Fallback: If no noun is found (e.g., just an adjective like "cancerous"),
            # just grab the very last word right before the tag.
            fallback_term = raw_phrase.split()[-1].strip("'\"")
            if fallback_term:
                extracted_terms.append(fallback_term)

    return extracted_terms

# ==========================================
# Data Processing Pipeline
# ==========================================



new_df['trad_metaphor_human_2'] = df['human_annotations_2'].apply(lambda x: extract_smart_terms(x, 'TRAD_METAPHOR'))
new_df['sci_term_human_2'] = df['human_annotations_2'].apply(lambda x: extract_smart_terms(x, 'SCI_TERM'))

# 3. Save the dataframe for review
output_path = "Annotations_Comparison.csv"
new_df.to_csv(output_path, index=False)
print(f"Data saved to {output_path}")


global_trad_human = list(set([term for sublist in new_df['trad_metaphor_human'] if isinstance(sublist, list) for term in sublist]))
global_sci_human = list(set([term for sublist in new_df['sci_term_human_2'] if isinstance(sublist, list) for term in sublist]))

print("\n--- Extraction Summary ---")
print(f"Total Unique GPT TRAD_METAPHORs:   {len(global_trad_gpt)}")
print(f"Total Unique Human TRAD_METAPHORs: {len(global_trad_human)}")
print(f"Total Unique GPT SCI_TERMs:        {len(global_sci_gpt)}")
print(f"Total Unique Human SCI_TERMs:      {len(global_sci_human)}")

In [7]:
def calculate_agreement(human_list, gpt_list):
    # Convert lists to sets for math operations, and lowercase to avoid case-sensitivity
    human_set = set(str(x).lower() for x in human_list)
    gpt_set = set(str(x).lower() for x in gpt_list)
    # print(human_set)

    # Calculate True Positives (Intersection)
    intersection = len(human_set.intersection(gpt_set))

    # Calculate Union
    union = len(human_set.union(gpt_set))

    # Jaccard Similarity
    jaccard = intersection / union if union > 0 else 0

    # Precision, Recall, F1
    precision = intersection / len(gpt_set) if len(gpt_set) > 0 else 0
    recall = intersection / len(human_set) if len(human_set) > 0 else 0

    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return pd.Series({'Jaccard': jaccard, 'Precision': precision, 'Recall': recall, 'F1_Score': f1})

# Apply the function to SCI_TERM
sci_agreement = new_df.apply(lambda row: calculate_agreement(row['sci_term_human_2'], row['sci_term_gpt']), axis=1)

# Apply the function to TRAD_METAPHORa
trad_agreement = new_df.apply(lambda row: calculate_agreement(row['trad_metaphor_human_2'], row['trad_metaphor_gpt']), axis=1)

print("Average SCI_TERM Agreement (F1-Score):", sci_agreement['F1_Score'].mean())
print("Average SCI_TERM Agreement (Precision):", sci_agreement['Precision'].mean())
print("Average SCI_TERM Agreement (Recall):", sci_agreement['Recall'].mean())

print("Average TRAD_METAPHOR Agreement (F1-Score):", trad_agreement['F1_Score'].mean())
print("Average TRAD_METAPHOR Agreement (Precision):", trad_agreement['Precision'].mean())
print("Average TRAD_METAPHOR Agreement (Recall):", trad_agreement['Recall'].mean())

Average SCI_TERM Agreement (F1-Score): 0.5288816689466483
Average SCI_TERM Agreement (Precision): 0.6137445887445887
Average SCI_TERM Agreement (Recall): 0.5385608985608985
Average TRAD_METAPHOR Agreement (F1-Score): 0.5931808278867103
Average TRAD_METAPHOR Agreement (Precision): 0.6407142857142858
Average TRAD_METAPHOR Agreement (Recall): 0.5683333333333334
